# NB-10 — Deployment (FastAPI + OpenWebUI)

**Goal:** Serve VisionMind via an OpenAI-compatible API and connect OpenWebUI.

---

In [ ]:
import sys
import json
import base64
import os
from io import BytesIO
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import requests
from dotenv import load_dotenv
from PIL import Image

load_dotenv(Path("..") / ".env")

API_BASE = os.getenv("API_BASE", "http://127.0.0.1:8000")
API_KEY = os.getenv("API_KEY", "")
HEADERS = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

## 1. Start the API locally

In a terminal (from repo root):

```bash
uvicorn src.serving.api_server:app --host 0.0.0.0 --port 8000
```

Or: `python -m src.serving.api_server`

In [ ]:
try:
    health = requests.get(f"{API_BASE}/health", timeout=5)
    print(health.status_code, health.json())
except requests.RequestException as exc:
    print(f"API not running: {exc}")
    print("Start uvicorn first, then re-run this cell.")

In [ ]:
payload = {
    "model": "visionmind-qwen2-vl",
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "max_tokens": 64,
}

RUN_LIVE = False  # set True when API + model are loaded

if RUN_LIVE:
    resp = requests.post(
        f"{API_BASE}/v1/chat/completions",
        headers={**HEADERS, "Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )
    print(resp.status_code)
    print(json.dumps(resp.json(), indent=2)[:800])
else:
    print("Set RUN_LIVE=True to call the live API.")

In [ ]:
def image_to_data_uri(img: Image.Image) -> str:
    buf = BytesIO()
    img.save(buf, format="JPEG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/jpeg;base64,{b64}"

if RUN_LIVE:
    img = Image.new("RGB", (224, 224), (80, 160, 220))
    uri = image_to_data_uri(img)
    mm_payload = {
        "model": "visionmind-qwen2-vl",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": "What color is dominant in this image?"},
                {"type": "image_url", "image_url": {"url": uri}},
            ],
        }],
        "max_tokens": 64,
    }
    resp = requests.post(
        f"{API_BASE}/v1/chat/completions",
        headers={**HEADERS, "Content-Type": "application/json"},
        json=mm_payload,
        timeout=300,
    )
    print(resp.json()["choices"][0]["message"]["content"])

## 2. Docker + OpenWebUI

```bash
cd VisionMind
cp .env.example .env   # set HF_TOKEN, API_KEY
docker compose -f docker/docker-compose.yml up --build
```

- API: http://localhost:8000
- OpenWebUI: http://localhost:3000

In OpenWebUI → **Settings → Connections** → add OpenAI-compatible endpoint:
- Base URL: `http://localhost:8000/v1` (or `http://api:8000/v1` inside compose network)
- API Key: same as `API_KEY` in `.env`

## 3. Performance notes

| Approach | Pros | Cons |
|----------|------|------|
| **This API (HF + uvicorn)** | Full multimodal, RAG, LoRA path | Slower than vLLM |
| **vLLM** | High throughput on GPU | Separate setup; vision support varies |
| **4-bit (BitsAndBytes)** | Fits 8GB GPU | Slight quality/latency tradeoff |
| **float16 on CUDA** | Faster than 4-bit on 16GB+ | Needs more VRAM |

For production throughput, consider vLLM behind the same OpenAI-compatible routes once Qwen2-VL vision is supported in your vLLM build.

## 4. Ollama alternative (optional)

Ollama works best with GGUF text models. For full Qwen2-VL multimodal + images, prefer this FastAPI stack or vLLM.

```bash
# Example: run a text-only model in Ollama and point OpenWebUI at http://localhost:11434/v1
ollama pull llama3.2
ollama serve
```

OpenWebUI can switch connections per model — use VisionMind API for image chat and Ollama for fast text-only tasks.

---

**Phase 7 complete.** Optional stretch: Phase 8 — image generation output (`NB-11`).